# 04 — Data Versioning

## What
Data versioning tracks changes to datasets over time, enabling reproducibility, rollback, and auditability. It is the ML equivalent of source control for code — except files are often too large for Git, so dedicated tools are needed.

## Why
Without data versioning, 'retrain on the same data' becomes impossible once preprocessing scripts or raw sources change. Regulators increasingly require auditability: which data was used to train the model that made *this* decision?

## Community context
DVC (Data Version Control, iterative.ai) is the dominant open-source solution. It stores data in S3/GCS/Azure while keeping a small `.dvc` pointer file in Git — giving you Git-native workflows without stuffing gigabytes into the repo. LakeFormation (AWS) and Delta Lake (Databricks) solve the same problem in warehouse contexts with ACID semantics.

In [ ]:
# DVC-style data versioning simulation
import hashlib, json, shutil, pathlib, time

class DVCStore:
    """
    Simulates DVC: stores data by content-hash, tracks pointers in .dvc files.
    In real DVC the cache backend is S3/GCS; here we use /tmp.
    """
    def __init__(self, cache_dir='/tmp/dvc_cache'):
        self.cache = pathlib.Path(cache_dir)
        self.cache.mkdir(parents=True, exist_ok=True)

    def _hash(self, data_bytes):
        return hashlib.md5(data_bytes).hexdigest()

    def add(self, filepath):
        """Stage a file: copy to content-addressed cache, write .dvc pointer"""
        fp = pathlib.Path(filepath)
        data = fp.read_bytes()
        h = self._hash(data)
        # Store in cache
        cache_path = self.cache / h[:2] / h[2:]
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if not cache_path.exists():
            shutil.copy(fp, cache_path)
        # Write .dvc pointer (what goes into Git)
        dvc_file = str(fp) + '.dvc'
        pointer = {
            'outs': [{'md5': h, 'path': str(fp.name), 'size': len(data)}],
            'created': time.strftime('%Y-%m-%dT%H:%M:%S')
        }
        pathlib.Path(dvc_file).write_text(json.dumps(pointer, indent=2))
        print(f'Tracked {fp.name} -> md5:{h[:8]}... ({len(data)} bytes)')
        return h

    def checkout(self, dvc_file, dest=None):
        """Restore a file from cache using its .dvc pointer"""
        pointer = json.loads(pathlib.Path(dvc_file).read_text())
        h = pointer['outs'][0]['md5']
        cache_path = self.cache / h[:2] / h[2:]
        out_path = dest or pointer['outs'][0]['path']
        shutil.copy(cache_path, out_path)
        print(f'Checked out md5:{h[:8]}... -> {out_path}')
        return out_path

    def status(self, filepath):
        """Check if current file matches tracked version"""
        dvc_file = str(filepath) + '.dvc'
        if not pathlib.Path(dvc_file).exists():
            return 'untracked'
        pointer = json.loads(pathlib.Path(dvc_file).read_text())
        tracked_hash = pointer['outs'][0]['md5']
        current_hash = self._hash(pathlib.Path(filepath).read_bytes())
        return 'modified' if current_hash != tracked_hash else 'unchanged'

# Demo workflow
import pandas as pd, numpy as np
dvc = DVCStore()

# v1: original dataset
df_v1 = pd.DataFrame({'x': np.random.randn(100), 'y': np.random.randint(0,2,100)})
df_v1.to_csv('/tmp/train.csv', index=False)
h1 = dvc.add('/tmp/train.csv')

print(f'Status after add: {dvc.status("/tmp/train.csv")}')

# v2: new data arrives, update dataset
df_v2 = pd.concat([df_v1, pd.DataFrame({'x': np.random.randn(20), 'y': np.random.randint(0,2,20)})]).reset_index(drop=True)
df_v2.to_csv('/tmp/train.csv', index=False)
print(f'Status after update: {dvc.status("/tmp/train.csv")}')
h2 = dvc.add('/tmp/train.csv')

print(f'\nv1 hash: {h1[:16]}  v2 hash: {h2[:16]}')
print('Different hashes confirm versions are distinct')

## DVC Pipeline DAGs

DVC pipelines define a DAG of stages: each stage has `deps` (inputs), `cmd` (transform command), and `outs` (outputs). This makes the entire data-to-model pipeline reproducible with `dvc repro`.

In [ ]:
# Simulated DVC pipeline definition (dvc.yaml structure)
pipeline = {
    'stages': {
        'preprocess': {
            'cmd': 'python preprocess.py --input data/raw.csv --output data/processed.csv',
            'deps': ['data/raw.csv', 'preprocess.py'],
            'outs': ['data/processed.csv']
        },
        'featurize': {
            'cmd': 'python featurize.py --input data/processed.csv --output data/features.parquet',
            'deps': ['data/processed.csv', 'featurize.py'],
            'outs': ['data/features.parquet']
        },
        'train': {
            'cmd': 'python train.py --features data/features.parquet --output models/model.pkl',
            'deps': ['data/features.parquet', 'train.py', 'params.yaml'],
            'outs': ['models/model.pkl'],
            'metrics': ['metrics/eval.json']
        },
    }
}

def topological_sort(stages):
    """Simple topo sort for pipeline DAG"""
    deps_map = {}
    outs_to_stage = {}
    for name, stage in stages.items():
        for out in stage.get('outs', []) + stage.get('metrics', []):
            outs_to_stage[out] = name
    for name, stage in stages.items():
        upstream = set()
        for dep in stage.get('deps', []):
            if dep in outs_to_stage:
                upstream.add(outs_to_stage[dep])
        deps_map[name] = upstream
    order = []
    visited = set()
    def visit(n):
        if n in visited: return
        visited.add(n)
        for dep in deps_map[n]:
            visit(dep)
        order.append(n)
    for n in stages:
        visit(n)
    return order

order = topological_sort(pipeline['stages'])
print('Pipeline execution order:')
for i, stage in enumerate(order, 1):
    s = pipeline['stages'][stage]
    print(f'  {i}. {stage}')
    print(f'     cmd: {s["cmd"][:60]}...' if len(s['cmd'])>60 else f'     cmd: {s["cmd"]}')